
# Assignment 1: Assessing Bicycle Lane Quality

## Team 1: Raina Allkoçi, Xinran Zhi, Rahma El Sayed

**Objective:** Classify bicycle-lane conditions as smooth or bumpy using smartphone accelerometer, gyroscope, and gravity measurements.

## 1. Data Loading and Understanding

### 1.1 Imports and dataset location

In [ ]:
import sys
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)


working_dir = Path.cwd()
project_folder = (working_dir if (working_dir / "raw data file").is_dir()
                  else working_dir.parent)
data_folder = project_folder / "raw data file"
if not data_folder.is_dir():
    raise FileNotFoundError(f"Raw data folder not found: {data_folder}")

zip_files = sorted(data_folder.rglob("*.zip"))
if not zip_files:
    raise FileNotFoundError(f"No ZIP recordings found in {data_folder}")
print("ZIP recordings found:", len(zip_files))

Python: 3.12.12
NumPy: 2.5.3
Pandas: 3.0.5
Matplotlib: 3.11.1
ZIP recordings found: 30


### 1.2 Load and combine the sensor recordings

For each ZIP, load the three requested sensors and verify that their timestamps align before combining them. Participant, road-condition, and recording identifiers are retained. Original ZIP files remain unchanged.

In [ ]:
def load_recording(zip_path):

    """Read one ZIP recording and align its three sensor files by timestamp."""
    
    with ZipFile(zip_path, "r") as archive:
        with archive.open("Accelerometer.csv") as file:
            acc = pd.read_csv(file)
        with archive.open("Gyroscope.csv") as file:
            gyro = pd.read_csv(file)
        with archive.open("Gravity.csv") as file:
            gravity = pd.read_csv(file)

    if not (acc["time"].equals(gyro["time"])
            and acc["time"].equals(gravity["time"])):
        raise ValueError(f"Sensor timestamps do not match: {zip_path}")

    return pd.DataFrame({
        "time": acc["time"],
        "seconds_elapsed": acc["seconds_elapsed"],
        "acc_x": acc["x"], "acc_y": acc["y"], "acc_z": acc["z"],
        "gyro_x": gyro["x"], "gyro_y": gyro["y"], "gyro_z": gyro["z"],
        "gravity_x": gravity["x"], "gravity_y": gravity["y"],
        "gravity_z": gravity["z"]
    })

In [27]:
all_recordings = []
for zip_path in zip_files:
    participant = zip_path.parent.parent.name
    road_condition = zip_path.parent.name.lower()

    recording = load_recording(zip_path)
    recording["participant"] = participant
    recording["road_condition"] = road_condition
    #Unique 
    recording["recording_id"] = zip_path.relative_to(data_folder).as_posix()
    all_recordings.append(recording)

combined_df = pd.concat(all_recordings, ignore_index=True)
print("Combined dataset shape:", combined_df.shape)
print("Number of recordings:", combined_df["recording_id"].nunique())
print("Number of participants:", combined_df["participant"].nunique())
display(combined_df.head())

Combined dataset shape: (420506, 14)
Number of recordings: 30
Number of participants: 3


,time,seconds_elapsed,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,gravity_x,gravity_y,gravity_z,participant,road_condition,recording_id
0,1789734532680360400,0.053360,0.525098,0.198844,-0.317852,0.576124,-0.386222,-0.125969,-0.723667,4.737252,8.556000,participant_1,bumpy,participant_1/bumpy/p1_bumpy_1.zip
1,1789734532690368800,0.063369,0.499797,0.149029,-0.292189,0.611401,-0.374717,-0.186475,-0.698366,4.787067,8.530338,participant_1,bumpy,participant_1/bumpy/p1_bumpy_1.zip
2,1789734532700376800,0.073377,0.565957,0.457386,0.393400,0.628788,-0.322669,-0.266257,-0.679233,4.838439,8.502855,participant_1,bumpy,participant_1/bumpy/p1_bumpy_1.zip
3,1789734532710384600,0.083385,0.593435,0.282436,0.727975,0.666647,-0.203669,-0.363661,-0.671845,4.891135,8.473240,participant_1,bumpy,participant_1/bumpy/p1_bumpy_1.zip
4,1789734532720392700,0.093393,0.555283,-0.118405,0.713513,0.723867,-0.047215,-0.479859,-0.681876,4.947210,8.439819,participant_1,bumpy,participant_1/bumpy/p1_bumpy_1.zip


### 1.3 Dataset overview and quality checks

Summarize recording counts and durations. The road-condition labels apply to **the whole recording**, which means that recordings labeled 'bumpy' can still contain smooth sections.

In [29]:
recording_counts = (
    combined_df.groupby(["participant", "road_condition"])["recording_id"]
    .nunique().unstack(fill_value=0)
)
recording_durations = (
    combined_df.groupby(["participant", "road_condition", "recording_id"])
    ["seconds_elapsed"].agg(lambda values: values.max() - values.min())
    .reset_index(name="duration_seconds")
)
duration_summary = (
    recording_durations.groupby("road_condition")["duration_seconds"]
    .agg(["count", "sum", "mean", "min", "max"])
)

print("Recording counts by participant and condition:")
display(recording_counts)
print("Recording durations (seconds) by condition:")
display(duration_summary)

Recording counts by participant and condition:


road_condition,bumpy,smooth
participant,,
participant_1,7,3
participant_2,7,3
participant_3,6,4


Recording durations (seconds) by condition:


,count,sum,mean,min,max
road_condition,,,,,
bumpy,20,2720.861194,136.043060,34.333133,265.051933
smooth,10,1486.669629,148.666963,40.704864,292.451381


## 2. Exploratory Data Analysis

The raw sensor signals are examined to identify patterns and potential artifacts caused by smartphone handling.

The analysis also aims to identify the portions of each recording corresponding to actual cycling, which will be retained for subsequent preprocessing and feature extraction.

### 2.1 Visualize a Raw Recording